<a href="https://colab.research.google.com/github/anamikavio7-commits/PSB-analysis/blob/main/PSB_Tomato_Final_Colab_Analysis_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PSB Tomato Analysis

**Study:** Effect of phosphate-solubilizing bacteria on tomato growth  
**Varieties:** Srijana and BL-410

This notebook uses a simple research-statistics workflow:
1. Load and inspect the raw Excel data
2. Calculate descriptive statistics (mean, SD, n)
3. One-way ANOVA
4. Fisher's LSD post-hoc comparisons when ANOVA is significant
5. Germination percentage and Seedling Vigor Index (SVI)
6. Chi-square test for germination counts

**Important design note:** The germination assay used one plate per treatment, so seed-level observations are subsamples of one experimental unit. Germination-stage ANOVA/LSD results should therefore be interpreted as preliminary screening results.

In [1]:
# 1. Import libraries

import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Load the Excel file

In [2]:
# Uploadthe Excel file in Colab.
# The filename is:
# Tomato_PSB_RAW_DATA.xlsx

file_path = "Tomato_PSB_RAW_DATA.xlsx"

excel = pd.ExcelFile(file_path)
print("Sheets available:")
print(excel.sheet_names)

Sheets available:
['README', 'Germ_Srijana_RAW', 'Germ_BL410_RAW', 'Seedling_Srijana_RAW', 'Seedling_BL410_RAW']


In [3]:
# Read all four raw-data sheets

germ_srijana = pd.read_excel(file_path, sheet_name="Germ_Srijana_RAW")
germ_bl410 = pd.read_excel(file_path, sheet_name="Germ_BL410_RAW")
seed_srijana = pd.read_excel(file_path, sheet_name="Seedling_Srijana_RAW")
seed_bl410 = pd.read_excel(file_path, sheet_name="Seedling_BL410_RAW")

print("Data loaded.")
print("Germination Srijana:", germ_srijana.shape)
print("Germination BL-410:", germ_bl410.shape)
print("Seedling Srijana:", seed_srijana.shape)
print("Seedling BL-410:", seed_bl410.shape)

Data loaded.
Germination Srijana: (300, 8)
Germination BL-410: (300, 8)
Seedling Srijana: (90, 7)
Seedling BL-410: (90, 7)


## 3. Basic data inspection

In [4]:
# First few rows

display(germ_srijana.head())
display(germ_bl410.head())
display(seed_srijana.head())
display(seed_bl410.head())

,Variety,Treatment,Replicate,Root_cm,Shoot_cm,Total_Length_cm,FreshWeight_g,Germinated(Y/N)
0,Srijana,10 mM MgSo4,1,5.3000,5.1000,10.4000,0.0010,Y
1,Srijana,10 mM MgSo4,2,6.1000,5.0000,11.1000,0.0090,Y
2,Srijana,10 mM MgSo4,3,2.0000,4.5000,6.5000,0.0110,Y
3,Srijana,10 mM MgSo4,4,4.5000,5.0000,9.5000,0.0090,Y
4,Srijana,10 mM MgSo4,5,3.5000,5.9000,9.4000,0.0080,Y


,Variety,Treatment,Replicate,Root_cm,Shoot_cm,Total_Length_cm,FreshWeight_g,Germinated(Y/N)
0,BL 410,10 mM MgSo4,1,5.9000,4.5000,10.4000,0.0040,Y
1,BL 410,10 mM MgSo4,2,4.6000,4.3000,8.9000,0.0010,Y
2,BL 410,10 mM MgSo4,3,7.0000,7.0000,14.0000,0.0000,Y
3,BL 410,10 mM MgSo4,4,2.0000,1.5000,3.5000,0.0000,Y
4,BL 410,10 mM MgSo4,5,5.4000,5.1000,10.5000,0.0000,Y


,Variety,Treatment,Replication,Root_length,Shoot_length,Total_length,fresh_wt_of_plant
0,Srijana,10 mM MgSO4,1,2.1000,7.8000,9.9000,0.0200
1,Srijana,10 mM MgSO4,2,1.0000,5.0000,6.0000,0.0600
2,Srijana,10 mM MgSO4,3,2.0000,8.0000,10.0000,0.0800
3,Srijana,10 mM MgSO4,4,1.0000,7.0000,8.0000,0.0800
4,Srijana,10 mM MgSO4,5,1.5000,7.0000,8.5000,0.0800


,Variety,Treatment,Replication,Root_length,Shoot_length,Total_length,Fresh_weight
0,BL 410,10 mM MgSO4,1,3.5000,6.7000,10.2000,0.0600
1,BL 410,10 mM MgSO4,2,0.9000,6.4000,7.3000,0.0800
2,BL 410,10 mM MgSO4,3,3.1000,7.0000,10.1000,0.0700
3,BL 410,10 mM MgSO4,4,1.9000,8.0000,9.9000,0.0900
4,BL 410,10 mM MgSO4,5,3.0000,6.2000,9.2000,0.0400


In [5]:
# Structure and missing values

for name, df in {
    "Germination - Srijana": germ_srijana,
    "Germination - BL-410": germ_bl410,
    "Seedling - Srijana": seed_srijana,
    "Seedling - BL-410": seed_bl410
}.items():
    print("\n" + "="*60)
    print(name)
    print("="*60)
    print(df.info())
    print("\nMissing values:")
    print(df.isna().sum())


Germination - Srijana
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Variety          300 non-null    object 
 1   Treatment        300 non-null    object 
 2   Replicate        300 non-null    int64  
 3   Root_cm          259 non-null    float64
 4   Shoot_cm         259 non-null    float64
 5   Total_Length_cm  259 non-null    float64
 6   FreshWeight_g    109 non-null    float64
 7   Germinated(Y/N)  300 non-null    object 
dtypes: float64(4), int64(1), object(3)
memory usage: 18.9+ KB
None

Missing values:
Variety              0
Treatment            0
Replicate            0
Root_cm             41
Shoot_cm            41
Total_Length_cm     41
FreshWeight_g      191
Germinated(Y/N)      0
dtype: int64

Germination - BL-410
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   

In [6]:
# Descriptive overview

print("Germination - Srijana")
display(germ_srijana.describe(include="all"))

print("Germination - BL-410")
display(germ_bl410.describe(include="all"))

print("Seedling - Srijana")
display(seed_srijana.describe(include="all"))

print("Seedling - BL-410")
display(seed_bl410.describe(include="all"))

Germination - Srijana


,Variety,Treatment,Replicate,Root_cm,Shoot_cm,Total_Length_cm,FreshWeight_g,Germinated(Y/N)
count,300,300,300.0000,259.0000,259.0000,259.0000,109.0000,300
unique,1,10,NaN,NaN,NaN,NaN,NaN,2
top,Srijana,10 mM MgSo4,NaN,NaN,NaN,NaN,NaN,Y
freq,300,30,NaN,NaN,NaN,NaN,NaN,259
mean,NaN,NaN,15.5000,5.4589,5.4936,10.9525,0.0117,NaN
std,NaN,NaN,8.6699,2.3366,1.7952,3.5631,0.0117,NaN
min,NaN,NaN,1.0000,0.0000,0.0000,0.2000,0.0000,NaN
25%,NaN,NaN,8.0000,4.0500,4.6500,9.4000,0.0010,NaN
50%,NaN,NaN,15.5000,5.8000,5.8000,11.5000,0.0090,NaN
75%,NaN,NaN,23.0000,7.0500,6.7000,13.5000,0.0200,NaN


Germination - BL-410


,Variety,Treatment,Replicate,Root_cm,Shoot_cm,Total_Length_cm,FreshWeight_g,Germinated(Y/N)
count,300,300,300.0000,168.0000,168.0000,168.0000,93.0000,300
unique,1,10,NaN,NaN,NaN,NaN,NaN,2
top,BL 410,10 mM MgSo4,NaN,NaN,NaN,NaN,NaN,Y
freq,300,30,NaN,NaN,NaN,NaN,NaN,168
mean,NaN,NaN,15.5000,5.1229,4.2152,9.3381,0.0124,NaN
std,NaN,NaN,8.6699,2.3428,1.5136,3.3999,0.0097,NaN
min,NaN,NaN,1.0000,0.0000,0.0000,0.2000,0.0000,NaN
25%,NaN,NaN,8.0000,3.3750,3.4000,7.2750,0.0010,NaN
50%,NaN,NaN,15.5000,5.4000,4.2000,9.9500,0.0150,NaN
75%,NaN,NaN,23.0000,7.0000,5.1000,11.5250,0.0190,NaN


Seedling - Srijana


,Variety,Treatment,Replication,Root_length,Shoot_length,Total_length,fresh_wt_of_plant
count,90,90,90.0000,90.0000,90.0000,90.0000,90.0000
unique,1,6,NaN,NaN,NaN,NaN,NaN
top,Srijana,10 mM MgSO4,NaN,NaN,NaN,NaN,NaN
freq,90,15,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,8.0000,1.5322,7.1167,8.6489,0.0590
std,NaN,NaN,4.3447,0.4822,0.8486,1.1033,0.0263
min,NaN,NaN,1.0000,0.5000,5.0000,6.0000,0.0100
25%,NaN,NaN,4.0000,1.2000,6.6000,8.0000,0.0400
50%,NaN,NaN,8.0000,1.4000,7.1000,8.5000,0.0600
75%,NaN,NaN,12.0000,1.8750,7.6000,9.3750,0.0700


Seedling - BL-410


,Variety,Treatment,Replication,Root_length,Shoot_length,Total_length,Fresh_weight
count,90,90,90.0000,90.0000,90.0000,90.0000,90.0000
unique,1,6,NaN,NaN,NaN,NaN,NaN
top,BL 410,10 mM MgSO4,NaN,NaN,NaN,NaN,NaN
freq,90,15,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,8.0000,1.6033,6.6567,8.2600,0.0691
std,NaN,NaN,4.3447,0.5836,0.9592,1.2285,0.0265
min,NaN,NaN,1.0000,0.8000,4.0000,5.2000,0.0100
25%,NaN,NaN,4.0000,1.1000,6.0000,7.4250,0.0500
50%,NaN,NaN,8.0000,1.4500,6.8000,8.3000,0.0700
75%,NaN,NaN,12.0000,2.0000,7.2750,9.0750,0.0900


## 4. Helper functions

In [7]:
def descriptive_stats(df, treatment_col, response_col):
    """Return n, mean and SD for each treatment."""
    result = (
        df.groupby(treatment_col)[response_col]
        .agg(n="count", mean="mean", SD="std")
        .reset_index()
    )
    return result


def one_way_anova(df, treatment_col, response_col):
    """Run one-way ANOVA across treatment groups."""
    groups = [
        group[response_col].dropna().values
        for _, group in df.groupby(treatment_col)
    ]

    F_stat, p_value = stats.f_oneway(*groups)

    return F_stat, p_value


def fisher_lsd(df, treatment_col, response_col, alpha=0.05):
    """Fisher's LSD pairwise comparison using the ANOVA pooled error term."""
    clean = df[[treatment_col, response_col]].dropna().copy()

    grouped = clean.groupby(treatment_col)[response_col]

    means = grouped.mean()
    ns = grouped.count()

    # Pooled within-group error from one-way ANOVA
    SSE = sum(
        ((group - group.mean()) ** 2).sum()
        for _, group in grouped
    )

    N = len(clean)
    k = len(means)
    df_error = N - k
    MSE = SSE / df_error

    treatments = list(means.index)
    rows = []

    t_critical = stats.t.ppf(1 - alpha / 2, df_error)

    for i in range(len(treatments)):
        for j in range(i + 1, len(treatments)):
            a = treatments[i]
            b = treatments[j]

            diff = means[a] - means[b]
            SE = np.sqrt(MSE * (1/ns[a] + 1/ns[b]))
            LSD = t_critical * SE

            t_value = diff / SE
            p_value = 2 * stats.t.sf(abs(t_value), df_error)

            rows.append({
                "Treatment 1": a,
                "Treatment 2": b,
                "Mean difference": diff,
                "LSD": LSD,
                "p-value": p_value,
                "Significant": p_value < alpha
            })

    return pd.DataFrame(rows), MSE, df_error


def run_anova_and_lsd(df, treatment_col, response_col, alpha=0.05):
    """Print ANOVA and run LSD only when ANOVA is significant."""
    print(f"\n{response_col}")
    print("-" * 50)

    F_stat, p_value = one_way_anova(df, treatment_col, response_col)

    print(f"F-statistic = {F_stat:.4f}")
    print(f"p-value = {p_value:.6g}")

    if p_value < alpha:
        print("ANOVA is significant (p < 0.05). Running Fisher's LSD...")
        lsd_table, MSE, df_error = fisher_lsd(
            df, treatment_col, response_col, alpha
        )
        display(lsd_table)
        return F_stat, p_value, lsd_table
    else:
        print("ANOVA is not significant (p >= 0.05). No LSD grouping performed.")
        return F_stat, p_value, None

## 5. Seedling assay — descriptive statistics and one-way ANOVA

The seedling assay has six treatments with 15 independent replicates per treatment.

ANOVA is run separately for each variety and response variable:
- Root length
- Shoot length
- Fresh weight

In [8]:
# Srijana seedling assay

print("===== SRIJANA SEEDLING ASSAY =====")

for variable in ["Root_length", "Shoot_length", "fresh_wt_of_plant"]:
    print(f"\nDescriptive statistics: {variable}")
    display(descriptive_stats(seed_srijana, "Treatment", variable))

    run_anova_and_lsd(
        seed_srijana,
        "Treatment",
        variable
    )

===== SRIJANA SEEDLING ASSAY =====

Descriptive statistics: Root_length


,Treatment,n,mean,SD
0,10 mM MgSO4,15,1.5000,0.3910
1,Compost,15,1.8933,0.3863
2,PSB-I 119,15,1.6133,0.6334
3,PSB-I 145,15,1.4000,0.3443
4,PSB-P,15,1.1267,0.2604
5,Vermicompost,15,1.6600,0.4778



Root_length
--------------------------------------------------
F-statistic = 5.4145
p-value = 0.000229617
ANOVA is significant (p < 0.05). Running Fisher's LSD...


,Treatment 1,Treatment 2,Mean difference,LSD,p-value,Significant
0,10 mM MgSO4,Compost,-0.3933,0.3134,0.0145,True
1,10 mM MgSO4,PSB-I 119,-0.1133,0.3134,0.4741,False
2,10 mM MgSO4,PSB-I 145,0.1000,0.3134,0.5275,False
3,10 mM MgSO4,PSB-P,0.3733,0.3134,0.0202,True
4,10 mM MgSO4,Vermicompost,-0.1600,0.3134,0.3130,False
5,Compost,PSB-I 119,0.2800,0.3134,0.0793,False
6,Compost,PSB-I 145,0.4933,0.3134,0.0024,True
7,Compost,PSB-P,0.7667,0.3134,0.0000,True
8,Compost,Vermicompost,0.2333,0.3134,0.1425,False
9,PSB-I 119,PSB-I 145,0.2133,0.3134,0.1795,False



Descriptive statistics: Shoot_length


...


Shoot_length
--------------------------------------------------
F-statistic = 3.7354
p-value = 0.00420844
ANOVA is significant (p < 0.05). Running Fisher's LSD...


,Treatment 1,Treatment 2,Mean difference,LSD,p-value,Significant
0,10 mM MgSO4,Compost,-0.6667,0.5737,0.0233,True
1,10 mM MgSO4,PSB-I 119,-0.0467,0.5737,0.8719,False
2,10 mM MgSO4,PSB-I 145,-0.6000,0.5737,0.0406,True
3,10 mM MgSO4,PSB-P,0.3800,0.5737,0.1913,False
4,10 mM MgSO4,Vermicompost,-0.1667,0.5737,0.5650,False
5,Compost,PSB-I 119,0.6200,0.5737,0.0345,True
6,Compost,PSB-I 145,0.0667,0.5737,0.8178,False
7,Compost,PSB-P,1.0467,0.5737,0.0005,True
8,Compost,Vermicompost,0.5000,0.5737,0.0867,False
9,PSB-I 119,PSB-I 145,-0.5533,0.5737,0.0585,False



Descriptive statistics: fresh_wt_of_plant


,Treatment,n,mean,SD
0,10 mM MgSO4,15,0.0540,0.0220
1,Compost,15,0.0693,0.0315
2,PSB-I 119,15,0.0493,0.0240
3,PSB-I 145,15,0.0667,0.0309
4,PSB-P,15,0.0507,0.0228
5,Vermicompost,15,0.0640,0.0213



fresh_wt_of_plant
--------------------------------------------------
F-statistic = 1.7111
p-value = 0.140918
ANOVA is not significant (p >= 0.05). No LSD grouping performed.


In [9]:
# BL-410 seedling assay

print("===== BL-410 SEEDLING ASSAY =====")

for variable in ["Root_length", "Shoot_length", "Fresh_weight"]:
    print(f"\nDescriptive statistics: {variable}")
    display(descriptive_stats(seed_bl410, "Treatment", variable))

    run_anova_and_lsd(
        seed_bl410,
        "Treatment",
        variable
    )

===== BL-410 SEEDLING ASSAY =====

Descriptive statistics: Root_length


,Treatment,n,mean,SD
0,10 mM MgSO4,15,1.8467,0.8626
1,Compost,15,1.4867,0.4138
2,PSB-I 119,15,1.6267,0.4464
3,PSB-I 145,15,1.7933,0.6628
4,PSB-P,15,1.5200,0.5102
5,Vermicompost,15,1.3467,0.3944



Root_length
--------------------------------------------------
F-statistic = 1.6659
p-value = 0.15179
ANOVA is not significant (p >= 0.05). No LSD grouping performed.

Descriptive statistics: Shoot_length


,Treatment,n,mean,SD
0,10 mM MgSO4,15,6.6800,0.5596
1,Compost,15,7.1600,0.8382
2,PSB-I 119,15,6.0133,1.0816
3,PSB-I 145,15,6.7200,1.2616
4,PSB-P,15,6.8200,0.7103
5,Vermicompost,15,6.5467,0.8975



Shoot_length
--------------------------------------------------
F-statistic = 2.5141
p-value = 0.035906
ANOVA is significant (p < 0.05). Running Fisher's LSD...


,Treatment 1,Treatment 2,Mean difference,LSD,p-value,Significant
0,10 mM MgSO4,Compost,-0.4800,0.6686,0.1571,False
1,10 mM MgSO4,PSB-I 119,0.6667,0.6686,0.0507,False
2,10 mM MgSO4,PSB-I 145,-0.0400,0.6686,0.9056,False
3,10 mM MgSO4,PSB-P,-0.1400,0.6686,0.6782,False
4,10 mM MgSO4,Vermicompost,0.1333,0.6686,0.6927,False
5,Compost,PSB-I 119,1.1467,0.6686,0.0010,True
6,Compost,PSB-I 145,0.4400,0.6686,0.1942,False
7,Compost,PSB-P,0.3400,0.6686,0.3148,False
8,Compost,Vermicompost,0.6133,0.6686,0.0717,False
9,PSB-I 119,PSB-I 145,-0.7067,0.6686,0.0386,True



Descriptive statistics: Fresh_weight


,Treatment,n,mean,SD
0,10 mM MgSO4,15,0.0633,0.0235
1,Compost,15,0.0733,0.0247
2,PSB-I 119,15,0.0633,0.0238
3,PSB-I 145,15,0.0727,0.0258
4,PSB-P,15,0.0680,0.0276
5,Vermicompost,15,0.0740,0.0346



Fresh_weight
--------------------------------------------------
F-statistic = 0.5064
p-value = 0.770641
ANOVA is not significant (p >= 0.05). No LSD grouping performed.


## 6. Germination assay

For root and shoot length, only germinated seeds are included because non-germinated seeds have no measurable length.

Germination-stage results are treated as **screening results** because each treatment was applied to a single plate.

In [10]:
# Keep only germinated seeds for length measurements

germ_srijana_germinated = germ_srijana[
    germ_srijana["Germinated(Y/N)"].astype(str).str.strip().str.upper() == "Y"
].copy()

germ_bl410_germinated = germ_bl410[
    germ_bl410["Germinated(Y/N)"].astype(str).str.strip().str.upper() == "Y"
].copy()

print("Germinated observations — Srijana:", len(germ_srijana_germinated))
print("Germinated observations — BL-410:", len(germ_bl410_germinated))

Germinated observations — Srijana: 259
Germinated observations — BL-410: 168


In [11]:
# Germination-stage descriptive statistics and ANOVA — Srijana

print("===== SRIJANA GERMINATION ASSAY =====")

for variable in ["Root_cm", "Shoot_cm"]:
    print(f"\nDescriptive statistics: {variable}")
    display(descriptive_stats(germ_srijana_germinated, "Treatment", variable))

    run_anova_and_lsd(
        germ_srijana_germinated,
        "Treatment",
        variable
    )

===== SRIJANA GERMINATION ASSAY =====

Descriptive statistics: Root_cm


,Treatment,n,mean,SD
0,10 mM MgSo4,24,4.1521,1.9726
1,GBO3,23,6.5587,2.1028
2,PSB(Prarambha),26,5.1077,2.4917
3,PSB-I 119,25,5.7320,1.8296
4,PSB-I 137,28,5.2518,2.8571
5,PSB-I 140,29,5.0345,2.4710
6,PSB-I 141,26,5.4192,2.7701
7,PSB-I 145,28,6.3250,2.2200
8,PSB-I 65,24,5.3958,1.7213
9,PSB-I 89,26,5.6423,1.9954



Root_cm
--------------------------------------------------
F-statistic = 2.1670
p-value = 0.024869
ANOVA is significant (p < 0.05). Running Fisher's LSD...


,Treatment 1,Treatment 2,Mean difference,LSD,p-value,Significant
0,10 mM MgSo4,GBO3,-2.4066,1.3163,0.0004,True
1,10 mM MgSo4,PSB(Prarambha),-0.9556,1.2770,0.1418,False
2,10 mM MgSo4,PSB-I 119,-1.5799,1.2892,0.0165,True
3,10 mM MgSo4,PSB-I 137,-1.0997,1.2549,0.0856,False
4,10 mM MgSo4,PSB-I 140,-0.8824,1.2449,0.1639,False
5,10 mM MgSo4,PSB-I 141,-1.2671,1.2770,0.0518,False
6,10 mM MgSo4,PSB-I 145,-2.1729,1.2549,0.0008,True
7,10 mM MgSo4,PSB-I 65,-1.2437,1.3023,0.0611,False
8,10 mM MgSo4,PSB-I 89,-1.4902,1.2770,0.0224,True
9,GBO3,PSB(Prarambha),1.4510,1.2913,0.0278,True



Descriptive statistics: Shoot_cm


,Treatment,n,mean,SD
0,10 mM MgSo4,24,5.0771,1.8598
1,GBO3,23,5.2978,1.4438
2,PSB(Prarambha),26,5.4154,1.9579
3,PSB-I 119,25,6.0840,1.3456
4,PSB-I 137,28,5.6054,1.7672
5,PSB-I 140,29,5.6517,2.2014
6,PSB-I 141,26,5.0769,2.1680
7,PSB-I 145,28,5.1143,1.6318
8,PSB-I 65,24,5.5167,1.6605
9,PSB-I 89,26,6.0692,1.5722



Shoot_cm
--------------------------------------------------
F-statistic = 1.1133
p-value = 0.353708
ANOVA is not significant (p >= 0.05). No LSD grouping performed.


In [12]:
# Germination-stage descriptive statistics and ANOVA — BL-410

print("===== BL-410 GERMINATION ASSAY =====")

for variable in ["Root_cm", "Shoot_cm"]:
    print(f"\nDescriptive statistics: {variable}")
    display(descriptive_stats(germ_bl410_germinated, "Treatment", variable))

    run_anova_and_lsd(
        germ_bl410_germinated,
        "Treatment",
        variable
    )

===== BL-410 GERMINATION ASSAY =====

Descriptive statistics: Root_cm


,Treatment,n,mean,SD
0,10 mM MgSo4,16,4.8063,2.2275
1,GBO3,19,6.3421,2.0500
2,PSB(Prarambha),14,4.1000,2.0987
3,PSB-I 119,20,6.1300,1.8599
4,PSB-I 137,16,5.2438,1.9089
5,PSB-I 140,21,3.1381,1.8704
6,PSB-I 141,16,6.8438,1.5646
7,PSB-I 145,20,4.0075,1.9575
8,PSB-I 65,12,4.2500,2.6169
9,PSB-I 89,14,6.6286,2.3724



Root_cm
--------------------------------------------------
F-statistic = 6.9849
p-value = 1.84252e-08
ANOVA is significant (p < 0.05). Running Fisher's LSD...


,Treatment 1,Treatment 2,Mean difference,LSD,p-value,Significant
0,10 mM MgSo4,GBO3,-1.5359,1.3652,0.0277,True
1,10 mM MgSo4,PSB(Prarambha),0.7063,1.4725,0.3449,False
2,10 mM MgSo4,PSB-I 119,-1.3237,1.3496,0.0545,False
3,10 mM MgSo4,PSB-I 137,-0.4375,1.4226,0.5444,False
4,10 mM MgSo4,PSB-I 140,1.6682,1.3352,0.0147,True
5,10 mM MgSo4,PSB-I 141,-2.0375,1.4226,0.0053,True
6,10 mM MgSo4,PSB-I 145,0.7988,1.3496,0.2442,False
7,10 mM MgSo4,PSB-I 65,0.5563,1.5365,0.4757,False
8,10 mM MgSo4,PSB-I 89,-1.8223,1.4725,0.0156,True
9,GBO3,PSB(Prarambha),2.2421,1.4172,0.0021,True



Descriptive statistics: Shoot_cm


,Treatment,n,mean,SD
0,10 mM MgSo4,16,4.2062,1.6229
1,GBO3,19,4.3947,1.4676
2,PSB(Prarambha),14,2.8571,1.2327
3,PSB-I 119,20,4.6600,1.1980
4,PSB-I 137,16,4.5062,1.4803
5,PSB-I 140,21,4.3524,1.5035
6,PSB-I 141,16,4.8937,0.8744
7,PSB-I 145,20,4.1375,1.6831
8,PSB-I 65,12,3.6083,1.9925
9,PSB-I 89,14,4.0214,1.4050



Shoot_cm
--------------------------------------------------
F-statistic = 2.3151
p-value = 0.0179203
ANOVA is significant (p < 0.05). Running Fisher's LSD...


,Treatment 1,Treatment 2,Mean difference,LSD,p-value,Significant
0,10 mM MgSo4,GBO3,-0.1885,0.9802,0.7046,False
1,10 mM MgSo4,PSB(Prarambha),1.3491,1.0572,0.0127,True
2,10 mM MgSo4,PSB-I 119,-0.4538,0.9689,0.3564,False
3,10 mM MgSo4,PSB-I 137,-0.3000,1.0214,0.5626,False
4,10 mM MgSo4,PSB-I 140,-0.1461,0.9586,0.7638,False
5,10 mM MgSo4,PSB-I 141,-0.6875,1.0214,0.1856,False
6,10 mM MgSo4,PSB-I 145,0.0687,0.9689,0.8887,False
7,10 mM MgSo4,PSB-I 65,0.5979,1.1032,0.2860,False
8,10 mM MgSo4,PSB-I 89,0.1848,1.0572,0.7303,False
9,GBO3,PSB(Prarambha),1.5376,1.0175,0.0033,True


## 7. Germination percentage

In [13]:
def germination_summary(df, treatment_col="Treatment", germination_col="Germinated(Y/N)"):
    temp = df.copy()
    temp[germination_col] = (
        temp[germination_col].astype(str).str.strip().str.upper()
    )

    summary = (
        temp.groupby(treatment_col)[germination_col]
        .agg(
            total="count",
            germinated=lambda x: (x == "Y").sum(),
            non_germinated=lambda x: (x == "N").sum()
        )
        .reset_index()
    )

    summary["GP (%)"] = summary["germinated"] / summary["total"] * 100
    return summary


print("Srijana germination percentage")
gp_srijana = germination_summary(germ_srijana)
display(gp_srijana)

print("BL-410 germination percentage")
gp_bl410 = germination_summary(germ_bl410)
display(gp_bl410)

Srijana germination percentage


,Treatment,total,germinated,non_germinated,GP (%)
0,10 mM MgSo4,30,24,6,80.0000
1,GBO3,30,23,7,76.6667
2,PSB(Prarambha),30,26,4,86.6667
3,PSB-I 119,30,25,5,83.3333
4,PSB-I 137,30,28,2,93.3333
5,PSB-I 140,30,29,1,96.6667
6,PSB-I 141,30,26,4,86.6667
7,PSB-I 145,30,28,2,93.3333
8,PSB-I 65,30,24,6,80.0000
9,PSB-I 89,30,26,4,86.6667


BL-410 germination percentage


,Treatment,total,germinated,non_germinated,GP (%)
0,10 mM MgSo4,30,16,14,53.3333
1,GBO3,30,19,11,63.3333
2,PSB(Prarambha),30,14,16,46.6667
3,PSB-I 119,30,20,10,66.6667
4,PSB-I 137,30,16,14,53.3333
5,PSB-I 140,30,21,9,70.0000
6,PSB-I 141,30,16,14,53.3333
7,PSB-I 145,30,20,10,66.6667
8,PSB-I 65,30,12,18,40.0000
9,PSB-I 89,30,14,16,46.6667


## 8. Chi-square test for germination counts

In [14]:
def chi_square_germination(df, treatment_col="Treatment",
                            germination_col="Germinated(Y/N)"):

    temp = df.copy()
    temp[germination_col] = (
        temp[germination_col].astype(str).str.strip().str.upper()
    )

    table = pd.crosstab(
        temp[treatment_col],
        temp[germination_col]
    )

    # Ensure both outcome columns exist
    for col in ["Y", "N"]:
        if col not in table.columns:
            table[col] = 0

    table = table[["Y", "N"]]

    chi2, p_value, dof, expected = stats.chi2_contingency(table)

    print("Observed counts:")
    display(table)

    print(f"Chi-square = {chi2:.4f}")
    print(f"Degrees of freedom = {dof}")
    print(f"p-value = {p_value:.6g}")

    return table, chi2, p_value, dof, expected


print("===== SRIJANA =====")
chi_srijana = chi_square_germination(germ_srijana)

print("\n===== BL-410 =====")
chi_bl410 = chi_square_germination(germ_bl410)

===== SRIJANA =====
Observed counts:


Germinated(Y/N),Y,N
Treatment,,
10 mM MgSo4,24,6
GBO3,23,7
PSB(Prarambha),26,4
PSB-I 119,25,5
PSB-I 137,28,2
PSB-I 140,29,1
PSB-I 141,26,4
PSB-I 145,28,2
PSB-I 65,24,6


Chi-square = 9.8597
Degrees of freedom = 9
p-value = 0.361964

===== BL-410 =====
Observed counts:


Germinated(Y/N),Y,N
Treatment,,
10 mM MgSo4,16,14
GBO3,19,11
PSB(Prarambha),14,16
PSB-I 119,20,10
PSB-I 137,16,14
PSB-I 140,21,9
PSB-I 141,16,14
PSB-I 145,20,10
PSB-I 65,12,18


Chi-square = 11.3095
Degrees of freedom = 9
p-value = 0.255088


## 9. Seedling Vigor Index (SVI)

In [15]:
# SVI = Germination percentage × mean seedling length
# Mean seedling length = mean(root length + shoot length) among germinated seeds

def calculate_svi(germ_df,
                  treatment_col="Treatment",
                  germination_col="Germinated(Y/N)",
                  root_col="Root_cm",
                  shoot_col="Shoot_cm"):

    temp = germ_df.copy()
    temp[germination_col] = (
        temp[germination_col].astype(str).str.strip().str.upper()
    )

    summary = (
        temp.groupby(treatment_col)
        .apply(
            lambda g: pd.Series({
                "GP (%)": (g[germination_col] == "Y").mean() * 100,
                "Mean seedling length (cm)": (
                    g.loc[g[germination_col] == "Y", root_col].mean()
                    + g.loc[g[germination_col] == "Y", shoot_col].mean()
                )
            }),
            include_groups=False
        )
        .reset_index()
    )

    summary["SVI"] = (
        summary["GP (%)"] *
        summary["Mean seedling length (cm)"]
    )

    return summary


print("===== SRIJANA SVI =====")
svi_srijana = calculate_svi(germ_srijana)
display(svi_srijana)

print("===== BL-410 SVI =====")
svi_bl410 = calculate_svi(germ_bl410)
display(svi_bl410)

===== SRIJANA SVI =====


,Treatment,GP (%),Mean seedling length (cm),SVI
0,10 mM MgSo4,80.0000,9.2292,738.3333
1,GBO3,76.6667,11.8565,909.0000
2,PSB(Prarambha),86.6667,10.5231,912.0000
3,PSB-I 119,83.3333,11.8160,984.6667
4,PSB-I 137,93.3333,10.8571,1013.3333
5,PSB-I 140,96.6667,10.6862,1033.0000
6,PSB-I 141,86.6667,10.4962,909.6667
7,PSB-I 145,93.3333,11.4393,1067.6667
8,PSB-I 65,80.0000,10.9125,873.0000
9,PSB-I 89,86.6667,11.7115,1015.0000


===== BL-410 SVI =====


,Treatment,GP (%),Mean seedling length (cm),SVI
0,10 mM MgSo4,53.3333,9.0125,480.6667
1,GBO3,63.3333,10.7368,680.0000
2,PSB(Prarambha),46.6667,6.9571,324.6667
3,PSB-I 119,66.6667,10.7900,719.3333
4,PSB-I 137,53.3333,9.7500,520.0000
5,PSB-I 140,70.0000,7.4905,524.3333
6,PSB-I 141,53.3333,11.7375,626.0000
7,PSB-I 145,66.6667,8.1450,543.0000
8,PSB-I 65,40.0000,7.8583,314.3333
9,PSB-I 89,46.6667,10.6500,497.0000


## 10. Final statistical interpretation rule

- **p < 0.05:** treatment effect is statistically significant; Fisher's LSD is performed.
- **p ≥ 0.05:** treatment effect is not statistically significant; no LSD grouping is performed.
- Germination percentage is evaluated with a chi-square test.
- Germination-stage inferential results should be described as preliminary screening because each treatment had one plate.

In [16]:
# Quick final summary of the key ANOVA results

results = []

analyses = [
    ("Srijana", "Germination", germ_srijana_germinated, "Root_cm"),
    ("Srijana", "Germination", germ_srijana_germinated, "Shoot_cm"),
    ("BL-410", "Germination", germ_bl410_germinated, "Root_cm"),
    ("BL-410", "Germination", germ_bl410_germinated, "Shoot_cm"),
    ("Srijana", "Seedling", seed_srijana, "Root_length"),
    ("Srijana", "Seedling", seed_srijana, "Shoot_length"),
    ("Srijana", "Seedling", seed_srijana, "fresh_wt_of_plant"),
    ("BL-410", "Seedling", seed_bl410, "Root_length"),
    ("BL-410", "Seedling", seed_bl410, "Shoot_length"),
    ("BL-410", "Seedling", seed_bl410, "Fresh_weight"),
]

for variety, stage, df, variable in analyses:
    F_stat, p_value = one_way_anova(df, "Treatment", variable)
    results.append({
        "Variety": variety,
        "Assay": stage,
        "Variable": variable,
        "F": F_stat,
        "p-value": p_value,
        "Significant (p<0.05)": p_value < 0.05
    })

anova_summary = pd.DataFrame(results)
display(anova_summary)

,Variety,Assay,Variable,F,p-value,Significant (p<0.05)
0,Srijana,Germination,Root_cm,2.1670,0.0249,True
1,Srijana,Germination,Shoot_cm,1.1133,0.3537,False
2,BL-410,Germination,Root_cm,6.9849,0.0000,True
3,BL-410,Germination,Shoot_cm,2.3151,0.0179,True
4,Srijana,Seedling,Root_length,5.4145,0.0002,True
5,Srijana,Seedling,Shoot_length,3.7354,0.0042,True
6,Srijana,Seedling,fresh_wt_of_plant,1.7111,0.1409,False
7,BL-410,Seedling,Root_length,1.6659,0.1518,False
8,BL-410,Seedling,Shoot_length,2.5141,0.0359,True
9,BL-410,Seedling,Fresh_weight,0.5064,0.7706,False


## 11. Notes for the manuscript

The statistical methods represented here correspond to the manuscript's stated analysis:
- completely randomized design
- one-way ANOVA for root length, shoot length and fresh weight
- Fisher's LSD at α = 0.05 after significant ANOVA
- chi-square test for germination counts
- germination-stage length analyses restricted to germinated seeds

The germination assay had one plate per treatment, so those observations are subsamples rather than independent replicated experimental units.